# API - 3 Machine Learnig Clasificación y Regresión

### Importamos las librerias necesarias

In [1]:
# Importamos las librerías necesarias
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

### Cargar el Data Set AmesHousing

In [2]:
# Cargamos el dataset AmesHousing.csv
data = pd.read_csv('AmesHousing.csv')
print("Dataset cargado. Dimensiones:", data.shape)
print(data.head())

Dataset cargado. Dimensiones: (2930, 82)
   Order        PID  MS SubClass MS Zoning  Lot Frontage  Lot Area Street  \
0      1  526301100           20        RL         141.0     31770   Pave   
1      2  526350040           20        RH          80.0     11622   Pave   
2      3  526351010           20        RL          81.0     14267   Pave   
3      4  526353030           20        RL          93.0     11160   Pave   
4      5  527105010           60        RL          74.0     13830   Pave   

  Alley Lot Shape Land Contour  ... Pool Area Pool QC  Fence Misc Feature  \
0   NaN       IR1          Lvl  ...         0     NaN    NaN          NaN   
1   NaN       Reg          Lvl  ...         0     NaN  MnPrv          NaN   
2   NaN       IR1          Lvl  ...         0     NaN    NaN         Gar2   
3   NaN       Reg          Lvl  ...         0     NaN    NaN          NaN   
4   NaN       IR1          Lvl  ...         0     NaN  MnPrv          NaN   

  Misc Val Mo Sold Yr Sold Sale T

### Acá realizo un preprocesamiento de datos

In [3]:
# Preprocesamiento de datos
# Seleccionamos la variable objetivo (SalePrice) y eliminamos columnas irrelevantes
target = 'SalePrice'
features = data.drop(columns=[target, 'Order', 'PID'])  # Eliminamos identificadores

# Identificamos columnas numéricas y categóricas
numeric_features = features.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = features.select_dtypes(include=['object']).columns.tolist()

# Manejo de valores faltantes
features[numeric_features] = features[numeric_features].fillna(features[numeric_features].median())
features[categorical_features] = features[categorical_features].fillna('Unknown')

# Preprocesador: escalado para numéricas, one-hot para categóricas
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# Aplicamos el preprocesador
X = preprocessor.fit_transform(features)
y = data[target]

print("Preprocesamiento completado. Forma de X:", X.shape)

Preprocesamiento completado. Forma de X: (2930, 319)


### Hago el Split del Data Set para los datos Train y Test

In [4]:
# Dividimos el dataset en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Conjuntos de datos divididos: Train shape:", X_train.shape, "Test shape:", X_test.shape)

Conjuntos de datos divididos: Train shape: (2344, 319) Test shape: (586, 319)


### Armo los tres modelos requeridos para comparar

In [5]:
# Definimos los modelos SVR
models = {
    'SVR Linear': SVR(kernel='linear'),
    'SVR RBF': SVR(kernel='rbf'),
    'SVR Poly': SVR(kernel='poly')
}

# Entrenamos y evaluamos cada modelo
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    results[name] = mae
    print(f"{name}: MAE = {mae:.4f}")

# Seleccionamos el modelo con el MAE más bajo
best_model = min(results, key=results.get)
print(f"\nEl mejor modelo es {best_model} con MAE = {results[best_model]:.4f}")

SVR Linear: MAE = 51391.2456
SVR RBF: MAE = 63736.6390
SVR Poly: MAE = 63739.1446

El mejor modelo es SVR Linear con MAE = 51391.2456


### Conclusiones
He implementado y evaluado tres variantes de SVR (lineal, RBF y polinómico) en el dataset Ames Housing. El modelo con el MAE más bajo es el recomendado para este análisis, evitando el sobreajuste observado en propuestas anteriores.